# Proceso de Machine Learning

## Preparación de datos

In [156]:
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import json
import numpy as np
from numpy._core.defchararray import upper

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import pickle
from pickle import dump

from sklearn.linear_model import LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.model_selection import GridSearchCV



# Leer el archivo CSV
df = pd.read_csv('../data/raw/sleep_mobile_stress_dataset_15000.csv', sep=',') # Este archivo CSV contiene comas como separadores
display(df.head())
print(df.columns) 
df.dtypes

,user_id,age,gender,occupation,daily_screen_time_hours,phone_usage_before_sleep_minutes,sleep_duration_hours,sleep_quality_score,stress_level,caffeine_intake_cups,physical_activity_minutes,notifications_received_per_day,mental_fatigue_score
0,1,56,Female,Designer,3.26,86,5.31,7.72,3.49,0,35,119,3.57
1,2,46,Female,Teacher,1.85,32,7.36,9.70,3.01,0,16,299,1.91
2,3,32,Female,Designer,3.04,107,4.50,6.38,5.03,0,17,21,6.05
3,4,25,Male,Software Engineer,9.00,36,6.68,5.53,10.00,0,3,220,9.92
4,5,38,Female,Teacher,3.52,56,7.57,6.69,6.71,4,92,167,5.99


Index(['user_id', 'age', 'gender', 'occupation', 'daily_screen_time_hours',
       'phone_usage_before_sleep_minutes', 'sleep_duration_hours',
       'sleep_quality_score', 'stress_level', 'caffeine_intake_cups',
       'physical_activity_minutes', 'notifications_received_per_day',
       'mental_fatigue_score'],
      dtype='str')


user_id                               int64
age                                   int64
gender                                  str
occupation                              str
daily_screen_time_hours             float64
phone_usage_before_sleep_minutes      int64
sleep_duration_hours                float64
sleep_quality_score                 float64
stress_level                        float64
caffeine_intake_cups                  int64
physical_activity_minutes             int64
notifications_received_per_day        int64
mental_fatigue_score                float64
dtype: object

In [140]:
print(f"Dimensiones del dataframe: {df.shape}")
print(df.info())

Dimensiones del dataframe: (15000, 13)
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 13 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   user_id                           15000 non-null  int64  
 1   age                               15000 non-null  int64  
 2   gender                            15000 non-null  str    
 3   occupation                        15000 non-null  str    
 4   daily_screen_time_hours           15000 non-null  float64
 5   phone_usage_before_sleep_minutes  15000 non-null  int64  
 6   sleep_duration_hours              15000 non-null  float64
 7   sleep_quality_score               15000 non-null  float64
 8   stress_level                      15000 non-null  float64
 9   caffeine_intake_cups              15000 non-null  int64  
 10  physical_activity_minutes         15000 non-null  int64  
 11  notifications_received_per_day    15000

In [141]:
print(f"Valores null por columna: \n{df.isnull().sum()}")
print(f"Valores unicos por columna: \n{df.nunique()}")

Valores null por columna: 
user_id                             0
age                                 0
gender                              0
occupation                          0
daily_screen_time_hours             0
phone_usage_before_sleep_minutes    0
sleep_duration_hours                0
sleep_quality_score                 0
stress_level                        0
caffeine_intake_cups                0
physical_activity_minutes           0
notifications_received_per_day      0
mental_fatigue_score                0
dtype: int64
Valores unicos por columna: 
user_id                             15000
age                                    42
gender                                  3
occupation                              8
daily_screen_time_hours               901
phone_usage_before_sleep_minutes      120
sleep_duration_hours                  501
sleep_quality_score                   831
stress_level                          900
caffeine_intake_cups                    5
physical_activity

In [142]:
print("Las columnas del dataset son:")
for i, columns in enumerate(df.columns.tolist()):
    print(f'{i+1}.',columns)

Las columnas del dataset son:
1. user_id
2. age
3. gender
4. occupation
5. daily_screen_time_hours
6. phone_usage_before_sleep_minutes
7. sleep_duration_hours
8. sleep_quality_score
9. stress_level
10. caffeine_intake_cups
11. physical_activity_minutes
12. notifications_received_per_day
13. mental_fatigue_score


### Eliminamos columnas que no aporten datos relevantes

In [143]:
print("No hay posibles datos duplicados que representen un problema en el analisis")

df = df.drop(['user_id', 'gender', 'occupation', 'notifications_received_per_day'], axis=1, inplace=False)
print(df.shape)
print("\n".join([f'{i+1}. {columns} \n' for i,columns in enumerate(df.columns.to_list())]))

No hay posibles datos duplicados que representen un problema en el analisis
(15000, 9)
1. age 

2. daily_screen_time_hours 

3. phone_usage_before_sleep_minutes 

4. sleep_duration_hours 

5. sleep_quality_score 

6. stress_level 

7. caffeine_intake_cups 

8. physical_activity_minutes 

9. mental_fatigue_score 



## Feature engineering

In [144]:
FINAL_COLS = ["age", "daily_screen_time_hours", "phone_usage_before_sleep_minutes", "sleep_duration_hours", "sleep_quality_score", "stress_level", "caffeine_intake_cups", "physical_activity_minutes", "mental_fatigue_score"]
df = df[FINAL_COLS]
df.describe()

,age,daily_screen_time_hours,phone_usage_before_sleep_minutes,sleep_duration_hours,sleep_quality_score,stress_level,caffeine_intake_cups,physical_activity_minutes,mental_fatigue_score
count,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.000000,15000.00000,15000.000000,15000.000000
mean,38.488467,5.501528,59.708933,6.509683,6.246362,6.980247,1.99880,59.157133,6.873009
std,12.007970,2.600085,34.641858,1.452689,1.713644,2.749382,1.41459,34.525705,2.730482
min,18.000000,1.000000,0.000000,4.000000,1.000000,1.000000,0.00000,0.000000,1.000000
25%,28.000000,3.260000,29.000000,5.260000,5.000000,4.750000,1.00000,29.000000,4.700000
50%,38.000000,5.490000,60.000000,6.490000,6.250000,7.380000,2.00000,59.000000,7.380000
75%,49.000000,7.760000,90.000000,7.790000,7.500000,10.000000,3.00000,89.000000,9.450000
max,59.000000,10.000000,119.000000,9.000000,10.000000,10.000000,4.00000,119.000000,10.000000


In [145]:
total_data_CON_outliers = df.copy()
total_data_SIN_outliers = df.copy()

outliers_cols = ["daily_screen_time_hours", "phone_usage_before_sleep_minutes"]

def replace_outliers(column, df):
  col_stats = df[column].describe()
  col_iqr = col_stats["75%"] - col_stats["25%"]
  upper_limit = round(float(col_stats["75%"] + 1.5 * col_iqr), 2)
  lower_limit = round(float(col_stats["25%"] - 1.5 * col_iqr), 2)

  if lower_limit < 0: lower_limit = min(df[column])
  # Vamos a quitar los outliers superiores
  df[column] = df[column].apply(lambda x: x if (x <= upper_limit) else upper_limit)
  # Vamos a quitar los outliers inferiores
  df[column] = df[column].apply(lambda x: x if (x >= lower_limit) else lower_limit)
  return df.copy(), [lower_limit, upper_limit]

outliers_dict = {}
for column in outliers_cols:
  total_data_SIN_outliers, limits = replace_outliers(column, total_data_SIN_outliers)
  outliers_dict.update({column: limits})

with open("../data/processed/outliers_dict.json", "w") as f:
  json.dump(outliers_dict, f)

### Escalado de valores

In [146]:
predictoras = ["age", "daily_screen_time_hours", "phone_usage_before_sleep_minutes", "sleep_duration_hours", "sleep_quality_score", "stress_level", "caffeine_intake_cups", "physical_activity_minutes"]
target = "mental_fatigue_score"

X_CON = total_data_CON_outliers.drop(target, axis = 1)[predictoras]
X_SIN = total_data_SIN_outliers.drop(target, axis = 1)[predictoras]
y = total_data_CON_outliers[target]

X_train_CON_outliers, X_test_CON_outliers, y_train, y_test = train_test_split(X_CON, y, test_size = 0.2, random_state = 10)
X_train_SIN_outliers, X_test_SIN_outliers = train_test_split(X_SIN, test_size = 0.2, random_state = 10)

In [147]:
# Normalización

norm_CON_outliers = StandardScaler()

norm_CON_outliers.fit(X_train_CON_outliers)

X_train_CON_outliers_norm = norm_CON_outliers.transform(X_train_CON_outliers)
X_train_CON_outliers_norm = pd.DataFrame(X_train_CON_outliers_norm, index = X_train_CON_outliers.index, columns = predictoras)

X_test_CON_outliers_norm = norm_CON_outliers.transform(X_test_CON_outliers)
X_test_CON_outliers_norm = pd.DataFrame(X_test_CON_outliers_norm, index = X_test_CON_outliers.index, columns = predictoras)

# SIN OUTLIERS
norm_SIN_outliers = StandardScaler()
norm_SIN_outliers.fit(X_train_SIN_outliers)

X_train_SIN_outliers_norm = norm_SIN_outliers.transform(X_train_SIN_outliers)
X_train_SIN_outliers_norm = pd.DataFrame(X_train_SIN_outliers_norm, index = X_train_SIN_outliers.index, columns = predictoras)

X_test_SIN_outliers_norm = norm_SIN_outliers.transform(X_test_SIN_outliers)
X_test_SIN_outliers_norm = pd.DataFrame(X_test_SIN_outliers_norm, index = X_test_SIN_outliers.index, columns = predictoras)


# ESCALADO MIN-MAX (MINMAXIMIZACIÓN)

scaler_CON_outliers = MinMaxScaler()
scaler_CON_outliers.fit(X_train_CON_outliers)

X_train_CON_outliers_scal = scaler_CON_outliers.transform(X_train_CON_outliers)
X_train_CON_outliers_scal = pd.DataFrame(X_train_CON_outliers_scal, index = X_train_CON_outliers.index, columns = predictoras)

X_test_CON_outliers_scal = scaler_CON_outliers.transform(X_test_CON_outliers)
X_test_CON_outliers_scal = pd.DataFrame(X_test_CON_outliers_scal, index = X_test_CON_outliers.index, columns = predictoras)

# SIN OUTLIERS
scaler_SIN_outliers = MinMaxScaler()
scaler_SIN_outliers.fit(X_train_SIN_outliers)

X_train_SIN_outliers_scal = scaler_SIN_outliers.transform(X_train_SIN_outliers)
X_train_SIN_outliers_scal = pd.DataFrame(X_train_SIN_outliers_scal, index = X_train_SIN_outliers.index, columns = predictoras)

X_test_SIN_outliers_scal = scaler_SIN_outliers.transform(X_test_SIN_outliers)
X_test_SIN_outliers_scal = pd.DataFrame(X_test_SIN_outliers_scal, index = X_test_SIN_outliers.index, columns = predictoras)


# Guardado de los datasets resultantes
X_train_CON_outliers.to_excel("../data/processed/X_train_CON_outliers.xlsx", index = False)
X_train_CON_outliers_norm.to_excel("../data/processed/X_train_CON_outliers_norm.xlsx", index = False)
X_train_CON_outliers_scal.to_excel("../data/processed/X_train_CON_outliers_scal.xlsx", index = False)
X_train_SIN_outliers.to_excel("../data/processed/X_train_SIN_outliers.xlsx", index = False)
X_train_SIN_outliers_norm.to_excel("../data/processed/X_train_SIN_outliers_norm.xlsx", index = False)
X_train_SIN_outliers_scal.to_excel("../data/processed/X_train_SIN_outliers_scal.xlsx", index = False)

X_test_CON_outliers.to_excel("../data/processed/X_test_CON_outliers.xlsx", index = False)
X_test_CON_outliers_norm.to_excel("../data/processed/X_test_CON_outliers_norm.xlsx", index = False)
X_test_CON_outliers_scal.to_excel("../data/processed/X_test_CON_outliers_scal.xlsx", index = False)
X_test_SIN_outliers.to_excel("../data/processed/X_test_SIN_outliers.xlsx", index = False)
X_test_SIN_outliers_norm.to_excel("../data/processed/X_test_SIN_outliers_norm.xlsx", index = False)
X_test_SIN_outliers_scal.to_excel("../data/processed/X_test_SIN_outliers_scal.xlsx", index = False)

y_train.to_excel("../data/processed/y_train.xlsx", index = False)
y_test.to_excel("../data/processed/y_test.xlsx", index = False)

# SCALERS

with open("../models/norm_CON_outliers.pkl", "wb") as file:
  pickle.dump(norm_CON_outliers, file)
with open("../models/norm_SIN_outliers.pkl", "wb") as file:
  pickle.dump(norm_SIN_outliers, file)
with open("../models/scaler_CON_outliers.pkl", "wb") as file:
  pickle.dump(scaler_CON_outliers, file)
with open("../models/scaler_SIN_outliers.pkl", "wb") as file:
  pickle.dump(scaler_SIN_outliers, file)

## Proceso de Machine Learning

In [148]:
datasets = [X_train_CON_outliers,
    X_train_CON_outliers_norm,
    X_train_CON_outliers_scal,
    X_train_SIN_outliers,
    X_train_SIN_outliers_norm,
    X_train_SIN_outliers_scal
    ]

y_train = y_train.squeeze()

models = []
metrics = []

for dataset in datasets:

    model = LinearRegression() #ML class
    model.fit(dataset, y_train)
    y_pred = model.predict(dataset)
    metric = model.score(dataset, y_train) # Calculado por coeficiente de determinacion
    metrics.append(metric)
    models.append(model)

best_metric = max(metrics)
best_index = metrics.index(best_metric)
print(f"El mejor dataset tiene {best_metric} de coeficiente de determinación")

El mejor dataset tiene 0.9002917154355717 de coeficiente de determinación


## Regularización

In [149]:
diccionario_modelos = {
    "lasso_model": {
        "model": Lasso(random_state=10),
        "params": {
                       "fit_intercept": [True, False],
                       "alpha": [0.001, 0.01, 0.1, 1, 10, 100],
                       "max_iter":[100, 300, 400, 500],
                       "selection": ["cyclic", "random"],
                       "tol":[00.1, 0.0001, 0.000001]
                  }
    },

    "ridge_model": {
        "model": Ridge(random_state=10),
        "params": {
                       "fit_intercept": [True, False],
                       "alpha": [0.001, 0.01, 0.1, 1, 10, 100],
                       "max_iter":[100, 300, 400, 500],
                       "tol":[00.1, 0.0001, 0.000001],
                       "solver": ["auto", "svd", "cholesky", "lsqr", "sparse_cg", "sag", "saga", "lbfgs"]
                  }
    },

    "elasticNet_model": {
        "model": ElasticNet(random_state=10),
        "params": {
                       "fit_intercept": [True, False],
                       "alpha": [0.001, 0.01, 0.1, 1, 10, 100],
                       "max_iter":[100, 300, 400, 500],
                       "l1_ratio": [0, 0.5, 1],
                       "fit_intercept": [True, False],
                       "selection": ["cyclic", "random"],
                       "tol":[00.1, 0.0001, 0.000001]
                  }
    }                        
}

In [150]:
best_scored_model = 0
best_model_gr = ""
name_model = ""

for name, info in diccionario_modelos.items():
    grid = GridSearchCV(info["model"], info["params"], scoring="r2", n_jobs = -1)
    grid.fit(datasets[best_index], y_train)

    if best_scored_model < grid.best_score_:
        best_scored_model = grid.best_score_
        best_model_gr = grid.best_estimator_
        name_model = name

print("El mejor modelo es: ", name_model)
print("El mejor estimador es: ", best_model_gr)
print("La mejor puntuación es: ", best_scored_model)

d:\miniconda\envs\ml_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.244e+02, tolerance: 6.569e+01
  model = cd_fast.enet_coordinate_descent(
d:\miniconda\envs\ml_env\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
720 fits failed out of a total of 5760.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
720 fits failed with the following error:
Traceback (most recent call last):
  File "d:\miniconda\envs\ml_env\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    

El mejor modelo es:  ridge_model
El mejor estimador es:  Ridge(alpha=1, fit_intercept=False, max_iter=100, random_state=10, solver='sag')
La mejor puntuación es:  0.8999337462600779


d:\miniconda\envs\ml_env\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 4.477e+03, tolerance: 6.569e+01
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/RidgeCV instead.
  model = cd_fast.enet_coordinate_descent(


## Métricas finales

In [151]:
datasets_test = [X_test_CON_outliers,
    X_test_CON_outliers_norm,
    X_test_CON_outliers_scal,
    X_test_SIN_outliers,
    X_test_SIN_outliers_norm,
    X_test_SIN_outliers_scal
    ]

y_test = y_test.squeeze()

model_f = Ridge(alpha=1, fit_intercept=False, max_iter=100, random_state=10, solver='sag')
model_f.fit(datasets[best_index], y_train)

y_pred = model_f.predict(datasets[best_index])
metric_train = mean_squared_error(y_train, y_pred)
metric_R2_train = r2_score(y_train, y_pred)


y_pred = model_f.predict(datasets_test[best_index])
metric_test = mean_squared_error(y_test, y_pred)
metric_R2_test = r2_score(y_test, y_pred)


print(f"La mejor métrica MSE de nuestros datasets x_train son: {metric_train} y los x_test son: {metric_test}")
print(f"El mejor coeficiente de determinacion de nuestros datasets x_train son: {metric_R2_train} y los x_test son: {metric_R2_test}")

#Guardado de modelo

dump(model, open("../models/modelo_LR_Ridge_alp1_fit-inter0_max-it100_solv-sag.sav", "wb"))

La mejor métrica MSE de nuestros datasets x_train son: 0.7451984518783221 y los x_test son: 0.7264131569594591
El mejor coeficiente de determinacion de nuestros datasets x_train son: 0.9002850114408962 y los x_test son: 0.9015955955052217


In [ ]:
predict_row = [[56, 3.26, 86, 5.31, 7.72, 3.49, 0, 35]]
prediccion = model_f.predict(predict_row)
print(prediccion)

[3.63238203]


d:\miniconda\envs\ml_env\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but Ridge was fitted with feature names
  warnings.warn(
